# 학교명 후보 개수 검증 — `data/interim/preprocessing.csv` 기반

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "utils") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "utils"))

REPO_ROOT

WindowsPath('D:/Study/dongguk_university/dreampath')

In [2]:
import pandas as pd

df = pd.read_csv(REPO_ROOT / "data" / "interim" / "preprocessing.csv", encoding="utf-8-sig")
df["comment_noun"] = df["comment_noun"].fillna("")  # 명사가 하나도 안 남은 행은 NaN으로 읽히므로 방어
df.shape

(1000, 4)

In [3]:
df.columns

Index(['comment_id', 'comment', 'comment_clean_1', 'comment_noun'], dtype='str')

## 초/중/고/대/학교로 끝나는 명사 개수 (댓글당 학교 1개 가설 검증)

In [4]:
from preprocessing import extract_school_candidates, noun_count

df["school_candidate"] = df["comment_noun"].apply(extract_school_candidates)
df["school_candidate_count"] = df["comment_noun"].apply(noun_count)
df[["comment", "comment_noun", "school_candidate", "school_candidate_count"]].head(20)

,comment,comment_noun,school_candidate,school_candidate_count
0,동국대학교,동국대학교,동국대학교,1
1,이번엔 중동고 차례입니다 🍗,이번 중동고 차례,중동고,1
2,우리 반/동아리 대표로 서초초 신청합니다!,우리 반 동아리 대표 서초초 신청,서초초,1
3,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 친구 치킨 건국대,잠실중 건국대,2
4,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 친구 서초초 같이 보고 저 이화여대부속초로 신청,이화여대부속초 서초초 보고,3
5,학식 말고 치킨 먹고 싶어요 인하대학교,학식 치킨 인하대학교,인하대학교,1
6,서울공업고.. 오늘만 기다렸어요,서울공업 오늘,,0
7,서강대 학생들 모여라,서강대 학생들,서강대,1
8,동아리방에서 기다릴게요 서초중,동아리방 서초중,서초중,1
9,대치중 학생입니다 대치중 뽑아주세요,대치중 학생,대치중,1


In [5]:
# 가설 검증: 댓글당 학교명 후보가 정말 1개씩인지
df["school_candidate_count"].value_counts().sort_index()

school_candidate_count
0    109
1    790
2     83
3     18
Name: count, dtype: int64

## 1순위 ① — 후보 0건: 학교명을 아예 못 찾은 댓글

In [6]:
zero_df = df[df["school_candidate_count"] == 0]
print(f"{len(zero_df)}개 행")
zero_df[["comment_id", "comment", "comment_noun"]]

109개 행


,comment_id,comment,comment_noun
6,C0007,서울공업고.. 오늘만 기다렸어요,서울공업 오늘
27,C0028,서울고 축제 준비팀입니다 치킨이면 밤샘 가능,서울 축제 준비팀 치킨이면 밤샘 가능
50,C0051,축제 뒤풀이 치킨은 대치초로 부탁해요,축제 뒤풀이 치킨 대치초로 부탁
57,C0058,경기고 경기고 치킨 주세요!,경기 치킨
65,C0066,서울고 축제 준비팀입니다 치킨이면 밤샘 가능,서울 축제 준비팀 치킨이면 밤샘 가능
...,...,...,...
951,C0952,우리 반/동아리 대표로 서울고 신청합니다!,우리 반 동아리 대표 서울 신청
961,C0962,오늘 회의 많은데 경기고 치킨 지원 부탁드려요,오늘 회의 경기 치킨 지원
963,C0964,서초초도 참여 완료!,서초초도 참여 완료
970,C0971,오늘은 경기고! 다시 말해 경기고!,오늘 경기 다시 말


조사 때문에 삭제된 애들이 다수 보임
=> 'comment_clean_1(이모지/특수문자만 지우고 형태소 분석은 안 거친 원문)' 이거 기준으로 다시 채워넣기

HOW : '초','중','고','대'라는 글자가 감지되면 바로 잘라서 후보로 넣자
- 어차피 뒤에 gt_match를 통해서 precision을 높힐 예정이니 일단 recall을 높히자

In [7]:
from preprocessing import extract_school_candidates_fallback

zero_mask = df["school_candidate_count"] == 0

df.loc[zero_mask, "school_candidate"] = df.loc[zero_mask, "comment_clean_1"].apply(
    extract_school_candidates_fallback
)
df.loc[zero_mask, "school_candidate_count"] = df.loc[zero_mask, "school_candidate"].apply(
    lambda s: len(s.split()) if s else 0
)

df.loc[zero_mask, ["comment_id", "comment", "comment_clean_1", "school_candidate", "school_candidate_count"]]

,comment_id,comment,comment_clean_1,school_candidate,school_candidate_count
6,C0007,서울공업고.. 오늘만 기다렸어요,서울공업고 오늘만 기다렸어요,서울공업고,1
27,C0028,서울고 축제 준비팀입니다 치킨이면 밤샘 가능,서울고 축제 준비팀입니다 치킨이면 밤샘 가능,서울고,1
50,C0051,축제 뒤풀이 치킨은 대치초로 부탁해요,축제 뒤풀이 치킨은 대치초로 부탁해요,대치초,1
57,C0058,경기고 경기고 치킨 주세요!,경기고 경기고 치킨 주세요,경기고,1
65,C0066,서울고 축제 준비팀입니다 치킨이면 밤샘 가능,서울고 축제 준비팀입니다 치킨이면 밤샘 가능,서울고,1
...,...,...,...,...,...
951,C0952,우리 반/동아리 대표로 서울고 신청합니다!,우리 반 동아리 대표로 서울고 신청합니다,서울고,1
961,C0962,오늘 회의 많은데 경기고 치킨 지원 부탁드려요,오늘 회의 많은데 경기고 치킨 지원 부탁드려요,경기고,1
963,C0964,서초초도 참여 완료!,서초초도 참여 완료,서초,1
970,C0971,오늘은 경기고! 다시 말해 경기고!,오늘은 경기고 다시 말해 경기고,경기고,1


- "서초"같은 지역명에서 초로 끝나는 이슈 발견 => 다른 로직이 필요 
- 말고 처럼 진짜 조사인 경우는 못 잡아냄 

In [8]:
# 폴백 적용 후 분포 재확인
df["school_candidate_count"].value_counts().sort_index()

school_candidate_count
0      3
1    877
2     99
3     21
Name: count, dtype: int64

'서 강 대'처럼 한 글자씩 띄어 쓴 경우 => 여전히 0건인 행만 대상으로, 한 글자 토큰을 합친 뒤 다시 폴백 적용

In [9]:
from preprocessing import merge_spaced_syllables

still_zero_mask = df["school_candidate_count"] == 0

merged_text = df.loc[still_zero_mask, "comment_clean_1"].apply(merge_spaced_syllables)
df.loc[still_zero_mask, "school_candidate"] = merged_text.apply(extract_school_candidates_fallback)
df.loc[still_zero_mask, "school_candidate_count"] = df.loc[still_zero_mask, "school_candidate"].apply(
    lambda s: len(s.split()) if s else 0
)

df.loc[still_zero_mask, ["comment_id", "comment", "comment_clean_1", "school_candidate", "school_candidate_count"]]

,comment_id,comment,comment_clean_1,school_candidate,school_candidate_count
381,C0382,서 강 대!!! 치킨 부탁드려요,서 강 대 치킨 부탁드려요,서강대,1
570,C0571,서 울 대.. 오늘만 기다렸어요,서 울 대 오늘만 기다렸어요,서울대,1
736,C0737,건 국 대.. 오늘만 기다렸어요,건 국 대 오늘만 기다렸어요,건국대,1


In [10]:
# 두 번째 폴백 적용 후 분포 재확인
df["school_candidate_count"].value_counts().sort_index()

school_candidate_count
1    880
2     99
3     21
Name: count, dtype: int64

## 1순위 ② — 후보 2건 이상: 여러 학교명 후보가 잡힌 댓글

In [11]:
out_path = REPO_ROOT / "data" / "interim" / "school_candidate_counts.csv"
df[["comment_id", "comment", "comment_clean_1", "comment_noun", "school_candidate", "school_candidate_count"]].to_csv(
    out_path, index=False, encoding="utf-8-sig"
)
out_path

WindowsPath('D:/Study/dongguk_university/dreampath/data/interim/school_candidate_counts.csv')

In [12]:
multi_df = df[df["school_candidate_count"] >= 2]
print(f"{len(multi_df)}개 행")
multi_df[["comment_id", "comment", "comment_noun", "school_candidate", "school_candidate_count"]]

120개 행


,comment_id,comment,comment_noun,school_candidate,school_candidate_count
3,C0004,잠실중 친구가 놀러왔지만 치킨은 건국대로 주세요,잠실중 친구 치킨 건국대,잠실중 건국대,2
4,C0005,이화여대부속초 친구랑 서초초 친구 같이 보고 있는데 저는 이화여대부속초로 신청!,이화여대부속초 친구 서초초 같이 보고 저 이화여대부속초로 신청,이화여대부속초 서초초 보고,3
28,C0029,우리 학교는 건대 입니다,우리 학교 건대,학교 건대,2
30,C0031,대치초 동아리랑 이화여대부속초 동아리 같이 응원하지만 대표는 대치초,대치초 동아리 이화여대부속초 같이 응원하지 대표,대치초 이화여대부속초,2
32,C0033,중앙대 친구가 놀러왔지만 치킨은 인하대로 주세요,중앙대 친구 치킨 인하대,중앙대 인하대,2
...,...,...,...,...,...
976,C0977,우리 학교는 대치중 입니다,우리 학교 대치중,학교 대치중,2
978,C0979,성균관대 vs 중앙대 얘기하다가 결국 성균관대로 참여합니다,성균관대 vs 중앙대 얘기 결국 참여,성균관대 중앙대,2
988,C0989,서초초 친구랑 한양대 친구 같이 보고 있는데 저는 서초초로 신청!,서초초 친구 한양대 같이 보고 저 서초초로 신청,서초초 한양대 보고,3
992,C0993,우리 학교는 인하부중 입니다,우리 학교 인하부중,학교 인하부중,2


- "보고" 이런건 gT랑 비교해서 돌리면 다 사라질 것들 
- 이때 문제가 되는건 진짜 학교가 두번 나오는 경우인데 이건 gt를 돌린 이후에 진행 (어차피 gt 돌린 이후에도 문제들이 생길것이므로 한번에 처리)